# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'
!rm -rf /kaggle/working/*

In [2]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.8 MB/s eta 0:00:00


In [3]:

CH=10; H=W=30
ROOT = Path.cwd()
OUT=ROOT/'task093_static_build'
OUT.mkdir(exist_ok=True)
VALDIR=ROOT/'task093_structural_validation'
VALDIR.mkdir(exist_ok=True)

In [4]:
def valid_mask(x):
    return (x.sum(1,keepdim=True)>0).float()

class SeparatorProjectionStatic(nn.Module):
    def __init__(self, sep=5):
        super().__init__(); self.sep=sep
    def forward(self,x):
        active=valid_mask(x); sep=x[:,self.sep:self.sep+1]
        nonsep=((x[:,1:,:,:].sum(1,keepdim=True)-sep).clamp(0,1))*active
        row_count=sep.sum(3,keepdim=True); col_count=sep.sum(2,keepdim=True)
        maxrow=row_count.amax((2,3),keepdim=True); maxcol=col_count.amax((2,3),keepdim=True)
        horizontal=(maxrow>=maxcol).float(); vertical=1-horizontal
        row_has=(row_count>0).float(); col_has=(col_count>0).float()
        rows_before=torch.cumsum(row_has,dim=2); rows_after=torch.flip(torch.cumsum(torch.flip(row_has,dims=[2]),dim=2),dims=[2])
        top=(row_has*(rows_before<=1).float()).clamp(0,1)
        bottom=(row_has*(rows_after<=1).float()).clamp(0,1)
        cols_before=torch.cumsum(col_has,dim=3); cols_after=torch.flip(torch.cumsum(torch.flip(col_has,dims=[3]),dim=3),dims=[3])
        leftc=(col_has*(cols_before<=1).float()).clamp(0,1)
        rightc=(col_has*(cols_after<=1).float()).clamp(0,1)
        top_full=top*torch.ones_like(sep); bottom_full=bottom*torch.ones_like(sep)
        left_full=leftc*torch.ones_like(sep); right_full=rightc*torch.ones_like(sep)
        above_side=(torch.flip(torch.cumsum(torch.flip(top_full,dims=[2]),dim=2),dims=[2])>0).float()*(1-sep)*active
        below_side=(torch.cumsum(bottom_full,dim=2)>0).float()*(1-sep)*active
        left_side=(torch.flip(torch.cumsum(torch.flip(left_full,dims=[3]),dim=3),dims=[3])>0).float()*(1-sep)*active
        right_side=(torch.cumsum(right_full,dim=3)>0).float()*(1-sep)*active
        cnt_above=(nonsep*above_side).sum(2,keepdim=True)
        cnt_below=(nonsep*below_side).sum(2,keepdim=True)
        cnt_left=(nonsep*left_side).sum(3,keepdim=True)
        cnt_right=(nonsep*right_side).sum(3,keepdim=True)
        dist_above=torch.flip(torch.cumsum(torch.flip(above_side,dims=[2]),dim=2),dims=[2])
        dist_below=torch.cumsum(below_side,dim=2)
        dist_left=torch.flip(torch.cumsum(torch.flip(left_side,dims=[3]),dim=3),dims=[3])
        dist_right=torch.cumsum(right_side,dim=3)
        path_above=above_side*(dist_above<=cnt_above).float()*(cnt_above>0).float()
        path_below=below_side*(dist_below<=cnt_below).float()*(cnt_below>0).float()
        path_left=left_side*(dist_left<=cnt_left).float()*(cnt_left>0).float()
        path_right=right_side*(dist_right<=cnt_right).float()*(cnt_right>0).float()
        protr=(horizontal*(path_above+path_below)+vertical*(path_left+path_right)).clamp(0,1)*active
        out=[]; occupied=torch.zeros_like(active)
        for k in range(CH):
            if k==self.sep:
                ch=(sep+protr).clamp(0,1)*active
            elif k==0:
                ch=torch.zeros_like(active)
            else:
                ch=torch.zeros_like(active)
            out.append(ch); occupied=(occupied+ch).clamp(0,1)
        out[0]=(1-occupied)*active
        return torch.cat(out,1)

def onehot(grid):
    a=np.array(grid,dtype=np.int64); h,w=a.shape
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for k in range(CH): x[0,k,:h,:w]=(a==k)
    return x,h,w

def predict(sess, grid):
    x,h,w=onehot(grid)
    return sess.run(None, {'input':x})[0].argmax(1)[0,:h,:w]

def validate(sess, task):
    rep={}
    for sec in ['train','test','arc-gen']:
        exact=0; wrong=0
        for ex in task[sec]:
            y=predict(sess, ex['input']); out=np.array(ex['output'])
            exact+=int(np.array_equal(y,out)); wrong += int(np.sum(y!=out))
        rep[sec]={'exact':exact,'total':len(task[sec]),'wrong_pixels':wrong}
    return rep

def shape_sig(mask):
    pts=np.argwhere(mask)
    if len(pts)==0: return 'none'
    r0,c0=pts.min(0); norm=sorted((int(r-r0), int(c-c0)) for r,c in pts)
    return ';'.join(f'{r},{c}' for r,c in norm)

def features(ex, idx, sec, pred_ok=None):
    inp=np.array(ex['input']); out=np.array(ex['output'])
    sep=(inp==5)
    nonsep=(inp!=0)&(inp!=5)
    colors=sorted(set(inp[nonsep].flatten().tolist())) if nonsep.any() else []
    sep_rows=np.where(sep.sum(1)>0)[0]
    sep_cols=np.where(sep.sum(0)>0)[0]
    orient='horizontal' if len(sep_rows)>=len(sep_cols) else 'vertical'
    if orient=='horizontal':
        sep_thickness=len(sep_rows); sep_pos_bucket=f'rows_{int(sep_rows[0])}_{int(sep_rows[-1])}'
        marker_side_counts={'above': int(np.sum(nonsep[:sep_rows[0],:])), 'below': int(np.sum(nonsep[sep_rows[-1]+1:,:]))}
        aligned_cols=sorted(set(np.argwhere(nonsep)[:,1].tolist())) if nonsep.any() else []
        side_set=','.join(k for k,v in marker_side_counts.items() if v>0) or 'none'
    else:
        sep_thickness=len(sep_cols); sep_pos_bucket=f'cols_{int(sep_cols[0])}_{int(sep_cols[-1])}'
        marker_side_counts={'left': int(np.sum(nonsep[:,:sep_cols[0]])), 'right': int(np.sum(nonsep[:,sep_cols[-1]+1:]))}
        aligned_cols=sorted(set(np.argwhere(nonsep)[:,0].tolist())) if nonsep.any() else []
        side_set=','.join(k for k,v in marker_side_counts.items() if v>0) or 'none'
    marker_count=int(nonsep.sum())
    output_added=int(np.sum((out==5)&(inp!=5)))
    return {
        'section':sec,'index':idx,'exact':pred_ok,
        'grid_shape':f'{inp.shape[0]}x{inp.shape[1]}',
        'separator_orientation':orient,
        'separator_thickness':sep_thickness,
        'separator_position':sep_pos_bucket,
        'marker_colors': '|'.join(map(str,colors)) if colors else 'none',
        'marker_count':marker_count,
        'marker_count_bucket': '0' if marker_count==0 else ('1-3' if marker_count<=3 else ('4-6' if marker_count<=6 else '7+')),
        'side_set': side_set,
        'aligned_axis_count': len(aligned_cols),
        'aligned_axis_count_bucket': '0' if len(aligned_cols)==0 else ('1-3' if len(aligned_cols)<=3 else ('4-6' if len(aligned_cols)<=6 else '7+')),
        'output_added_count': output_added,
        'output_added_bucket': '0' if output_added==0 else ('1-5' if output_added<=5 else ('6-10' if output_added<=10 else '11+')),
    }

def structural_validation(sess, task):
    rows=[]
    for sec in ['train','test','arc-gen']:
        for i,ex in enumerate(task[sec]):
            pred=predict(sess, ex['input']); ok=int(np.array_equal(pred,np.array(ex['output'])))
            rows.append(features(ex,i,sec,ok))
    import csv
    per=VALDIR/'per_example_structural_features_and_accuracy.csv'
    with open(per,'w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)
    group_rows=[]
    feature_cols=[c for c in rows[0].keys() if c not in ('section','index','exact')]
    for col in feature_cols:
        vals=sorted(set(r[col] for r in rows))
        for v in vals:
            sub=[r for r in rows if r[col]==v]
            group_rows.append({'feature':col,'value':v,'exact':sum(int(r['exact']) for r in sub),'total':len(sub),'accuracy':sum(int(r['exact']) for r in sub)/len(sub)})
    group=VALDIR/'structural_group_accuracy.csv'
    with open(group,'w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=['feature','value','exact','total','accuracy'])
        writer.writeheader(); writer.writerows(group_rows)
    # holdout: values absent in visible train, evaluate on arc-gen
    train_rows=[r for r in rows if r['section']=='train']
    arc_rows=[r for r in rows if r['section']=='arc-gen']
    hold_rows=[]
    for col in feature_cols:
        train_vals=set(r[col] for r in train_rows)
        sub=[r for r in arc_rows if r[col] not in train_vals]
        if sub:
            hold_rows.append({'feature':col,'heldout_exact':sum(int(r['exact']) for r in sub),'heldout_total':len(sub),'accuracy':sum(int(r['exact']) for r in sub)/len(sub),'heldout_values_sample':'|'.join(map(str, sorted(set(r[col] for r in sub))[:20]))})
    hold=VALDIR/'structural_holdout_summary_direct_features.csv'
    with open(hold,'w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=['feature','heldout_exact','heldout_total','accuracy','heldout_values_sample'])
        writer.writeheader(); writer.writerows(hold_rows)
    return per,group,hold,hold_rows

In [5]:
task=json.load(open(Path(COMPETITION)/'task093.json'))
model=SeparatorProjectionStatic().eval()
model_path=ROOT/'task093_static_graph.onnx'
torch.onnx.export(model, torch.zeros(1,CH,H,W), str(model_path), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)

/tmp/ipykernel_16/3184528126.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1,CH,H,W), str(model_path), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)


In [6]:

# force older IR if possible
m=onnx.load(str(model_path)); m.ir_version=min(m.ir_version,7); onnx.save(m,str(model_path))
onnx.checker.check_model(str(model_path))
ops={}
for node in m.graph.node: ops[node.op_type]=ops.get(node.op_type,0)+1
forbidden=[op for op in ['Loop','Scan','NonZero','Unique','Script','Function'] if ops.get(op,0)]
risk=[op for op in ['Shape','Range','Expand','Gather','ScatterND','ConstantOfShape','Resize','NonMaxSuppression'] if ops.get(op,0)]
sess=ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
rep=validate(sess,task)


In [7]:
# 70/30 arc-gen split
arc=task['arc-gen']; idx=list(range(len(arc))); random.Random(0).shuffle(idx); test=set(idx[:max(1,int(round(0.3*len(idx))))])
fit_e=fit_t=test_e=test_t=0
for i,ex in enumerate(arc):
    ok=int(np.array_equal(predict(sess,ex['input']),np.array(ex['output'])))
    if i in test: test_e+=ok; test_t+=1
    else: fit_e+=ok; fit_t+=1
split={'seed':0,'fit_exact':fit_e,'fit_total':fit_t,'test_exact':test_e,'test_total':test_t}
per,group,hold,hold_rows=structural_validation(sess,task)
arch={'onnx_size_bytes':model_path.stat().st_size,'under_1_4mb':model_path.stat().st_size<1_400_000,'forbidden':forbidden,'dynamic_scatter_risk_ops':risk,'op_counts':ops,'input_shape':[1,10,30,30],'output_shape':[1,10,30,30]}
summary={'task':'task093','validation':rep,'split_report':split,'onnx':arch,'structural_holdouts':hold_rows}
(VALDIR/'task093_structural_validation_report.md').write_text('# task093 structural validation report\n\n' + json.dumps(summary,indent=2))
with open(ROOT/'task093_static_graph_summary.json','w') as f: json.dump(summary,f,indent=2)
sub=ROOT/'submission.zip'
with zipfile.ZipFile(sub,'w',zipfile.ZIP_DEFLATED) as zf: zf.write(model_path,'task093.onnx')


In [8]:
# zip report
import subprocess
repzip=ROOT/'task093_structural_validation_report.zip'
if repzip.exists(): repzip.unlink()
subprocess.check_call(['zip','-r',str(repzip),str(VALDIR)], stdout=subprocess.DEVNULL)
print(json.dumps(summary,indent=2))
print('submission',sub)
print('notebook',nb_path)
# asserts
assert rep['train']['exact']==rep['train']['total'] and rep['test']['exact']==rep['test']['total'] and rep['arc-gen']['exact']==rep['arc-gen']['total']
assert split['test_exact']==split['test_total']
assert arch['under_1_4mb'] and not forbidden and not risk

{
  "task": "task093",
  "validation": {
    "train": {
      "exact": 3,
      "total": 3,
      "wrong_pixels": 0
    },
    "test": {
      "exact": 1,
      "total": 1,
      "wrong_pixels": 0
    },
    "arc-gen": {
      "exact": 261,
      "total": 261,
      "wrong_pixels": 0
    }
  },
  "split_report": {
    "seed": 0,
    "fit_exact": 183,
    "fit_total": 183,
    "test_exact": 78,
    "test_total": 78
  },
  "onnx": {
    "onnx_size_bytes": 73617,
    "under_1_4mb": true,
    "forbidden": [],
    "dynamic_scatter_risk_ops": [],
    "op_counts": {
      "Constant": 137,
      "ReduceSum": 8,
      "Greater": 11,
      "Cast": 20,
      "Slice": 14,
      "Sub": 4,
      "Clip": 17,
      "Mul": 34,
      "ReduceMax": 2,
      "GreaterOrEqual": 1,
      "CumSum": 12,
      "LessOrEqual": 8,
      "Add": 13,
      "Concat": 1
    },
    "input_shape": [
      1,
      10,
      30,
      30
    ],
    "output_shape": [
      1,
      10,
      30,
      30
    ]
  },
  "struc

NameError: name 'nb_path' is not defined